# Identity Resolution Matcher - Wikidata x OurAirports (LLM, zero-shot)

### Purpose

Zero-shot LLM baseline for the same Wikidata x OurAirports matching problem that
[`identity_resolution_matcher.ipynb`](identity_resolution_matcher.ipynb) solves with a trained ML
classifier. No training step: each candidate pair's attributes go straight into a prompt, the
model judges match/no-match directly.

For the comparison to be fair, this notebook:
- evaluates on the **same 363 held-out `test` pairs** the ML notebook reports its metrics on
  (never the `train_val` pairs — there's no training here, so there's no leakage risk, but using
  a different slice would make the numbers incomparable),
- shows the model the **same attributes** the ML notebook's final feature set actually used:
  `name`, `city`, `countryName`, `countryCode`, `localCode`, `elevation`, `lat`, `lon` — **not**
  `icao`/`iata`, which the ML notebook deliberately excludes (`icao` is the identifier the gold
  standard's own labels were built from, so showing it would let either model "cheat" instead of
  actually matching on the ambiguous attributes).

Two models, both zero-shot: **`qwen2.5:7b` via local Ollama** (free, runs on this machine) and
*(not yet run)* a Claude model via the Anthropic API, for a cloud/capable-model comparison point.

## 1. Load the Test Set

In [1]:
import pandas as pd
from pathlib import Path
from PyDI.io import load_csv

DATA_DIR = Path("../../../data/normalized_outputs")
GOLD_STANDARD_DIR = Path("../../../data/identity_resolution/gold_standard")

df_wd = load_csv(DATA_DIR / "wikidataNormalized.csv", name="wikidata")
df_oa = load_csv(DATA_DIR / "ourairportsNormalized.csv", name="ourairports")
gold = pd.read_csv(GOLD_STANDARD_DIR / "gold_standard_wd_oa.csv")

test = gold[gold["split"] == "test"].reset_index(drop=True)
print(f"test pairs: {len(test):,}")
print(test["type"].value_counts())

test pairs: 363
type
hard positive            75
normal positive          75
hard negative            75
normal negative          75
heliport negative        40
code changed positive    20
heliport positive         3
Name: count, dtype: int64


## 2. Build the Pair Prompt

Same attribute set as the ML notebook's final ("after") feature set, shown as plain text instead
of hand-engineered similarity features — the LLM reasons over the raw values itself. Missing
values are shown as `(missing)` rather than left blank, so the model doesn't mistake "no value"
for "empty string value".

In [2]:
ATTRS = ["name", "city", "countryName", "countryCode", "localCode", "elevation", "lat", "lon"]

wd_i = df_wd.set_index("id")
oa_i = df_oa.set_index("id")


def format_record(row, attrs=ATTRS):
    lines = []
    for a in attrs:
        v = row.get(a)
        lines.append(f"{a}: {v if pd.notna(v) else '(missing)'}")
    return "\n".join(lines)


def build_pair_prompt(id1, id2):
    rec_a = format_record(wd_i.loc[id1])
    rec_b = format_record(oa_i.loc[id2])
    return (
        f"Record A (source: Wikidata):\n{rec_a}\n\n"
        f"Record B (source: OurAirports):\n{rec_b}\n\n"
        "Are Record A and Record B the same real-world airport?"
    )


print(build_pair_prompt(test.loc[0, "id1"], test.loc[0, "id2"]))

Record A (source: Wikidata):
name: tuxpan airport
city: mexico
countryName: mexico
countryCode: MX
localCode: (missing)
elevation: (missing)
lat: 19.5836520098
lon: -103.3833330901

Record B (source: OurAirports):
name: zapotiltic tuxpan airport
city: zapotiltic
countryName: mexico
countryCode: MX
localCode: ZPC
elevation: 1188.72
lat: 19.5996
lon: -103.371

Are Record A and Record B the same real-world airport?


## 3. Ollama Classifier (`qwen2.5:7b`, zero-shot)

Structured output via Ollama's `format` (JSON schema) parameter, so the response always parses —
no manual text scraping. `confidence` sometimes comes back as a 0-100 scale instead of 0-1 despite
the schema; normalized on parse. A response that fails to parse at all falls back to a flagged
"unknown" prediction rather than crashing the loop.

In [3]:
import json as jsonlib

import ollama

OLLAMA_MODEL = "qwen2.5:7b"
MAX_OUTPUT_TOKENS = 1200  # safety cap: a short JSON answer never needs more; caps a runaway generation

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "match": {"type": "boolean"},
        "confidence": {"type": "number"},
        "reasoning": {"type": "string"},
    },
    "required": ["match", "confidence", "reasoning"],
}

SYSTEM_PROMPT = (
    "You are an expert at airport entity matching. Decide whether two airport records, "
    "each from a different data source, refer to the same real-world airport. "
    "Sources differ systematically: each dataset covers a different subset of attributes, "
    "uses different spelling/abbreviations, and may report slightly different coordinates "
    "for the same airport. "
    "A missing value on either side is NOT evidence that the records are different - it "
    "usually just means that source doesn't track that attribute. Ignore any attribute "
    "missing on either side, and base your decision only on the attributes present on both "
    "records. "
    "Wikidata's `city` field is often a region/state/province name rather than an actual "
    "city or town (e.g. \"milne bay province\" instead of \"misima island\"), while "
    "OurAirports' `city` is usually the actual municipality. A city-level mismatch between "
    "sources does not necessarily mean the records are different airports - it may just "
    "reflect this difference in granularity. "
    "Weight the evidence roughly like this: "
    "`name` and coordinates (`lat`/`lon`) are the strongest signals - a `name` difference "
    "does NOT rule out a match, since each source uses different naming conventions "
    "(abbreviations like 'Intl'/'Rgnl'/'Muni', including or omitting the word 'Airport', "
    "translated or renamed names); treat name similarity as supporting evidence, not an "
    "exact-match requirement. "
    "A `countryCode` mismatch does NOT reliably indicate different airports: one side may "
    "use the sovereign state's code and the other the territory's own code for the same "
    "place (e.g. US/PR, FR/PF, GB/FK, CN/MO, MA/EH). Don't treat `countryCode` disagreement "
    "alone as a hard filter - weigh it together with name and coordinates. `countryName` is "
    "even less reliable since sources use different naming conventions (e.g. "
    "'Congo-Brazzaville', 'Russia', 'British Virgin Islands'); treat `countryName` agreement "
    "as weaker supporting evidence only. "
    "`city` is a weaker, noisier signal (see the granularity note above). "
    "`localCode` and `elevation` are secondary, confirming signals - useful when both sides "
    "have a value and they match or conflict, but their absence is common and not suspicious. "
    "A heliport and a nearby airport/aerodrome are usually DIFFERENT facilities, even when "
    "very close geographically or similarly named - heliports typically need a much tighter "
    "match (near-identical name/location) than regular airports to be considered the same. "
    "Keep your `reasoning` to 1-2 concise sentences - state only the deciding factors, do not "
    "restate every attribute one by one."
)


def classify_pair_ollama(id1, id2, model=OLLAMA_MODEL):
    prompt = build_pair_prompt(id1, id2)
    try:
        resp = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            format=RESPONSE_SCHEMA,
            options={"num_predict": MAX_OUTPUT_TOKENS},
        )
        parsed = jsonlib.loads(resp["message"]["content"])
        confidence = float(parsed["confidence"])
        if confidence > 1:
            confidence /= 100.0
        return {
            "pred": bool(parsed["match"]),
            "confidence": confidence,
            "reasoning": parsed["reasoning"],
            "parse_error": False,
        }
    except Exception as exc:
        return {"pred": None, "confidence": None, "reasoning": str(exc), "parse_error": True}


# sanity check on one pair
classify_pair_ollama(test.loc[0, "id1"], test.loc[0, "id2"])

{'pred': True,
 'confidence': 0.8,
 'reasoning': 'The records have similar names, matching coordinates, and the same country and city. The small discrepancies in name and coordinates do not rule out a match given the context.',
 'parse_error': False}

## 4. Run on a Small Sample First

`qwen2.5:7b` running locally isn't instant — before committing to all 363 pairs, time a small
sample to see what the full run would cost in wall-clock time.

In [4]:
import time

SEED = 1234
N_PER_TYPE = 3  # stratified sample: up to this many per `type`, so every category is represented

sample = (
    test.groupby("type", group_keys=False)
    .apply(lambda g: g.sample(min(N_PER_TYPE, len(g)), random_state=SEED))
    .reset_index(drop=True)
)
print(f"sample size: {len(sample)}")
print(sample["type"].value_counts())

SAMPLE_SIZE = len(sample)

t0 = time.time()
results = [classify_pair_ollama(r["id1"], r["id2"]) for _, r in sample.iterrows()]
elapsed = time.time() - t0

sample_results = pd.concat([sample.reset_index(drop=True), pd.DataFrame(results)], axis=1)
print(f"{SAMPLE_SIZE} pairs in {elapsed:.1f}s ({elapsed / SAMPLE_SIZE:.2f}s/pair) "
      f"-> full 363 pairs would take roughly {elapsed / SAMPLE_SIZE * 363 / 60:.1f} min")
print(f"parse errors: {sample_results['parse_error'].sum()}")
sample_results[["id1", "id2", "type", "label", "pred", "confidence", "reasoning"]]

/var/folders/1b/jxsrmkz17_qc4wdf17nykhbw0000gn/T/ipykernel_17857/835289984.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(N_PER_TYPE, len(g)), random_state=SEED))


sample size: 21
type
code changed positive    3
hard negative            3
hard positive            3
heliport negative        3
heliport positive        3
normal negative          3
normal positive          3
Name: count, dtype: int64
21 pairs in 58.7s (2.79s/pair) -> full 363 pairs would take roughly 16.9 min
parse errors: 0


,id1,id2,type,label,pred,confidence,reasoning
0,wd_6641,oa_82621,code changed positive,1,False,0.40,The name and location coordinates differ signi...
1,wd_7451,oa_82820,code changed positive,1,True,0.90,"Both records have the same 'name', 'countryNam..."
2,wd_21168,oa_81496,code changed positive,1,True,0.95,"The records have identical names, countries, a..."
3,wd_21750,oa_21438,hard negative,0,True,0.85,The name and coordinates strongly support a ma...
4,wd_16112,oa_21301,hard negative,0,True,0.85,The records have the same name variation ('roc...
5,wd_100,oa_25384,hard negative,0,False,0.80,"The names differ, with Record A being an airpo..."
6,wd_5422,oa_26515,hard positive,1,True,0.90,"Strong match on name, countryCode, and lat/lon..."
7,wd_15739,oa_30396,hard positive,1,True,0.90,"The records match on 'name', 'countryName', 'c..."
8,wd_1714,oa_28597,hard positive,1,True,0.95,"Both records have the same `name`, `countryCod..."
9,wd_4001,oa_59594,heliport negative,0,False,0.03,The records have different names and slightly ...


In [5]:
list(sample_results.iloc[[13],].reasoning)

['The records have the same name (similar enough, considering the spelling differences), localCode, elevation, and latitude/longitude are close, and both are in the United States. However, the city names differ significantly, and one record is specifically a heliport, which is typically not considered the same as an airport.}']

In [6]:
list(sample_results.iloc[[3],].reasoning)[0]

'The name and coordinates strongly support a match, despite slight differences. The local codes and elevations are missing and do not conflict.'

**Next steps, once the sample looks right**: bump `SAMPLE_SIZE` to run all 363 pairs, then
evaluate with the same `sklearn.metrics` the ML notebook uses (accuracy/precision/recall/f1/
confusion matrix) for a direct side-by-side comparison, and do error analysis on the pairs where
this disagrees with the gold label — reading `reasoning` on those should show whether it's
missing the same cases the ML model missed, or different ones entirely.

## 5. Run on All 363 Test Pairs

At ~15-20s/pair this takes roughly 1.5-2 hours - prints progress every 25 pairs so it's clear
it's still running, not stuck. Saves the full result set to a local CSV, since re-running from
scratch is expensive: reload it later instead of recomputing (`pd.read_csv(RESULTS_PATH)`).

In [7]:
full_results = []
t0 = time.time()
for i, (_, r) in enumerate(test.iterrows()):
    full_results.append(classify_pair_ollama(r["id1"], r["id2"]))
    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(test)} done ({(time.time() - t0) / 60:.1f} min elapsed)")
elapsed = time.time() - t0

full_results_df = pd.concat([test.reset_index(drop=True), pd.DataFrame(full_results)], axis=1)
print(f"{len(test)} pairs in {elapsed / 60:.1f} min ({elapsed / len(test):.2f}s/pair)")
print(f"parse errors: {full_results_df['parse_error'].sum()} / {len(full_results_df)} "
      f"({full_results_df['parse_error'].mean():.1%})")

LLM_OUTPUT_DIR = Path("../../../data/identity_resolution/LLM_output")
LLM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = LLM_OUTPUT_DIR / "llm_matcher_results_qwen2.5_7b.csv"
full_results_df.to_csv(RESULTS_PATH, index=False)
print(f"Saved to {RESULTS_PATH}")

  25/363 done (1.2 min elapsed)
  50/363 done (8.4 min elapsed)
  75/363 done (9.7 min elapsed)
  100/363 done (11.0 min elapsed)
  125/363 done (12.2 min elapsed)
  150/363 done (52.0 min elapsed)
  175/363 done (53.2 min elapsed)
  200/363 done (54.7 min elapsed)
  225/363 done (56.1 min elapsed)
  250/363 done (57.5 min elapsed)
  275/363 done (58.7 min elapsed)
  300/363 done (60.0 min elapsed)
  325/363 done (61.5 min elapsed)
  350/363 done (63.2 min elapsed)
363 pairs in 64.0 min (10.59s/pair)
parse errors: 0 / 363 (0.0%)
Saved to ../../../data/identity_resolution/LLM_output/llm_matcher_results_qwen2.5_7b.csv


## 6. Evaluate

Same metrics (and the same `sklearn.metrics` functions) as the ML notebook's Section 4, computed
only on rows with a valid prediction - a parse error has no `pred` to score, so it's reported
separately as a reliability number rather than counted as wrong.

In [8]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)

valid = full_results_df[full_results_df["pred"].notna()].copy()
n_valid, n_total = len(valid), len(full_results_df)
print(f"valid predictions: {n_valid} / {n_total} ({n_valid / n_total:.1%}); "
      f"{n_total - n_valid} parse errors excluded from the metrics below")

y_true = valid["label"]
y_pred = valid["pred"].astype(int)
# confidence is "how sure the model is in its own answer" - convert to P(match=True) for roc_auc
y_proba = valid["confidence"].where(valid["pred"], 1 - valid["confidence"])

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
metrics = pd.Series({
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred),
    "recall": recall_score(y_true, y_pred),
    "f1": f1_score(y_true, y_pred),
    "roc_auc": roc_auc_score(y_true, y_proba),
    "tp": tp, "fp": fp, "fn": fn, "tn": tn,
}, name="qwen2.5:7b (zero-shot)")
metrics

valid predictions: 363 / 363 (100.0%); 0 parse errors excluded from the metrics below


accuracy       0.909091
precision      0.884615
recall         0.930636
f1             0.907042
roc_auc        0.752677
tp           161.000000
fp            21.000000
fn            12.000000
tn           169.000000
Name: qwen2.5:7b (zero-shot), dtype: float64

### Breakdown by Pair Type

In [9]:
# same framing as the ML notebook: "accuracy within group" is recall for positive types
# and specificity for negative types
type_breakdown = (
    valid.assign(correct=lambda d: d["pred"] == (d["label"] == 1))
    .groupby("type")["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy_within_group"})
    .sort_values("accuracy_within_group")
)
type_breakdown

,accuracy_within_group,count
type,,
heliport positive,0.666667,3
hard negative,0.773333,75
code changed positive,0.850000,20
hard positive,0.933333,75
normal negative,0.946667,75
normal positive,0.960000,75
heliport negative,1.000000,40


### Error Analysis

Reading the `reasoning` on the mismatches should show whether this misses the same kind of
cases the ML model struggled with (e.g. `hard negative`/heliport pairs) or different ones
entirely. Parse errors are listed separately below (not scored as right or wrong).

In [8]:
import textwrap

errors = valid[valid["pred"] != (valid["label"] == 1)]
print(f"{len(errors)} errors out of {n_valid} valid predictions")

for _, row in errors.iterrows():
    print(f"{row['id1']} / {row['id2']} | type={row['type']} label={row['label']} "
          f"pred={row['pred']} confidence={row['confidence']:.2f}")
    print(textwrap.fill(row["reasoning"], width=80))
    print()

31 errors out of 363 valid predictions
wd_16949 / oa_24675 | type=hard positive label=1 pred=False confidence=0.65
The records have similar coordinates and the same country and city, supporting a
match. However, the name difference (vojens airport vs. skrydstrup air base) and
the slight elevation discrepancy (42.0 vs. 42.9768) suggest they are likely
different facilities, possibly a nearby air base.

wd_823 / oa_43639 | type=hard positive label=1 pred=False confidence=0.50
The names differ, with Record B including 'airfield' not in Record A, and the
coordinates and elevation mismatch suggest different locations.

wd_20825 / oa_62311 | type=hard positive label=1 pred=False confidence=0.50
The airports have similar names but different cities. The coordinates and
elevations do not strongly support a match, given the significant distance
apart.

wd_5167 / oa_82439 | type=hard positive label=1 pred=False confidence=0.60
The coordinates and names differ significantly, and the cities are slig

In [9]:
parse_errors = full_results_df[full_results_df["parse_error"]]
print(f"{len(parse_errors)} parse errors, by type:")
print(parse_errors["type"].value_counts())
parse_errors[["id1", "id2", "type", "label", "reasoning"]]

0 parse errors, by type:
Series([], Name: count, dtype: int64)


,id1,id2,type,label,reasoning


**Findings (v1 prompt, full 363-pair run)**:

**v1 prompt's `name` clause (for reference, to compare against v2 below)**:
```
`name` and coordinates (`lat`/`lon`) are the strongest signals - a `name` difference does
NOT rule out a match, since each source uses different naming conventions (abbreviations
like 'Intl'/'Rgnl'/'Muni', including or omitting the word 'Airport', translated or renamed
names); treat name similarity as supporting evidence, not an exact-match requirement.
```

- **Errors track `type`, not randomness.** Every false negative is a positive type (hard/
  normal/code-changed/heliport positive); every false positive is a negative type (hard/normal
  negative). The model fails specifically in the direction each type was designed to be
  deceptive in, not uniformly across the dataset.
- **`hard negative` is the single biggest failure bucket** (14 of 31 errors, ~45%): close
  coordinates + matching country/city consistently override a genuine name difference, e.g.
  `wycombe air park` vs. `raf high wycombe` (likely a civilian/military site pair) and
  `khabarovsk` vs. `khabarovsk-novy` ("novy" = new - an old/new airport pair). The model is too
  willing to explain away a name mismatch as "just a naming convention difference."
- **Likely root cause: the v1 system prompt's "a `name` difference does NOT rule out a match"
  guidance overcorrects.** It was added to stop abbreviation/spelling variants from being
  rejected, but it also makes the model dismiss *meaningful* name differences - exactly the
  pattern `hard negative` pairs are constructed around.
- **Asymmetric treatment of name variation.** `code changed positive` errors are all full
  renames (`rondonopolis airport` vs. `maestro marinho franco airport`), not abbreviations - the
  model tolerates abbreviation-level differences but not a full rename, even with matching
  coordinates/country, suggesting the "name doesn't have to match" instruction isn't
  generalizing to this case the way it does to abbreviations.
- **Next experiment**: rewrite the name-weighting guidance to distinguish *why* names differ
  (abbreviation/translation/rename vs. two genuinely different facilities) rather than treating
  any mismatch as equally forgivable, and re-run on this same 363 pairs to see if it shifts
  errors out of `hard negative` without reintroducing the abbreviation false negatives v1 fixed.
  Proposed **v2** replacement for the `name`-weighting clause in `SYSTEM_PROMPT`:
  ```
  Weigh a `name` difference by WHY it likely differs, not just whether it matches:
  - Spelling/abbreviation/translation variants, or a full rename of the same site, do NOT
    rule out a match on their own.
  - BUT treat it as a likely DIFFERENT facility - even with very close or identical
    coordinates - when one name has a qualifier the other lacks that signals a distinct
    facility or era: military terms ('air base', 'RAF'), 'junction', or 'new'/'novy'/'old'.
    These are the words hard-negative test cases are typically built around (a civilian/
    military split, or an old/new airport pair at the same site). Don't let close
    coordinates alone override this kind of qualifier mismatch.
  ```

## v2 Results vs. v1 (full 363-pair re-run)

v1 was re-run after v2 (same prompt, re-reverted) to get a complete confusion matrix rather than
the earlier reconstruction. Note: Ollama's default sampling isn't deterministic, so re-running
the identical v1 prompt doesn't reproduce the exact same errors each time (31 vs. 33 total on
two v1 runs) - the numbers below are from this second, complete v1 run.

| type (n) | v1 wrong | v2 wrong | change |
|---|---|---|---|
| hard positive (75) | 5 | 15 | worse (+10) |
| normal positive (75) | 3 | 6 | worse (+3) |
| code changed positive (20) | 3 | 5 | worse (+2) |
| hard negative (75) | 17 | 4 | much better (-13) |
| normal negative (75) | 4 | 1 | better (-3) |
| heliport positive (3) | 1 | 1 | same |
| heliport negative (40) | 0 | 0 | same |
| **Total** | **33** | **32** | **~flat** |

| metric | v1 | v2 |
|---|---|---|
| accuracy | 0.909 | 0.912 |
| precision | 0.885 | 0.967 |
| recall | 0.931 | 0.844 |
| f1 | 0.907 | 0.901 |
| roc_auc | 0.753 | 0.844 |

**Conclusion**: the v2 rule change ("only forgive a name difference with a shared root/token")
simultaneously fixed most of the `hard negative`/`normal negative` false positives it targeted
(-16 errors combined, boosting precision 0.885 -> 0.967 and roc_auc 0.753 -> 0.844) and
over-rejected legitimate renamed/abbreviated matches in the positive categories (+15 errors
combined, dropping recall 0.931 -> 0.844) - both are side effects of the same overly strict
"shared root" criterion, not one causing the other. Total error count is roughly unchanged
(33 -> 32), but the precision/recall balance shifted hard. This highlights a difficult trade-off
in LLM-based matching: balancing tolerance for legitimate name variation against generalizing
that tolerance too far is hard to tune with prompt wording alone. For now, to keep recall
higher, we default to the **v1** prompt rather than v2.

## The Underlying Tension: Rules vs. Free-Form Judgment

Same issue as `name`, different shape: discrete signals (`countryCode`, heliport keywords) can
get a crisp rule the model applies consistently; continuous signals (`elevation`, coordinates)
have no such anchor, so the model re-judges "big or small" every time with no fixed reference,
giving inconsistent calls on similar-sized gaps. ML avoids this - `NumericComparator` bakes in a
fixed scale for the classifier to learn a boundary from; the LLM has neither a threshold nor
training data to calibrate one. Explicit rules raise consistency but risk overcorrecting (v2);
free-form judgment keeps generalization but sacrifices consistency - prompt wording alone can't
resolve this for continuous attributes.